<!-- Cell 1 -->
Deliverable: unified, cleaned dataset

In [1]:
# Cell 2
import pandas as pd
import numpy as np
import re
import html

In [2]:
# Cell 3
from pathlib import Path
import sys

# Get the current working directory
PROJECT_ROOT = Path.cwd()

# If running from the notebooks folder, go up one level
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Add project root to Python's import path
sys.path.insert(0, str(PROJECT_ROOT))

# Import loader
from src.loader import load_csv_folder

# Dataset directory
ZENODO_DIR = PROJECT_ROOT / "data" / "raw" / "zenodo"

# put all csv files into single df
data_df = load_csv_folder(ZENODO_DIR)

#display data
data_df

Loading CEAS_08.csv...
Loaded 39154 rows
Loading Enron.csv...
Loaded 29767 rows
Loading Ling.csv...
Loaded 2859 rows
Loading Nazario.csv...
Loaded 1565 rows
Loading Nazario_5.csv...
Loaded 3065 rows
Loading Nigerian_5.csv...
Loaded 6331 rows
Loading Nigerian_Fraud.csv...
Loaded 3332 rows
Loading SpamAssasin.csv...
Loaded 5809 rows
Loading TREC_05.csv...
Loaded 55990 rows
Loading TREC_06.csv...
Loaded 16457 rows
Loading TREC_07.csv...
Loaded 53757 rows

All datasets combined
Total rows: 218086
Total columns: 8


,sender,receiver,date,subject,body,label,urls,source_file
0,Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1.0,1,CEAS_08.csv
1,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1.0,1,CEAS_08.csv
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1.0,1,CEAS_08.csv
3,Michael Parker <ivqrnai@pobox.com>,SpamAssassin Dev <xrh@spamassassin.apache.org>,"Tue, 05 Aug 2008 17:31:20 -0600",Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,0.0,1,CEAS_08.csv
4,Gretchen Suggs <externalsep1@loanofficertool.com>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 19:31:21 -0400",SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7...,1.0,1,CEAS_08.csv
...,...,...,...,...,...,...,...,...
218081,SCC <Gerry.Rossi4360@kinki-kids.com>,Deficient <deficient@flax9.uwaterloo.ca>,"Fri, 06 Jul 2007 06:53:36 -0400",Job: just for you.,\n\n\n\nWhile we may have high ...,1.0,1,TREC_07.csv
218082,Sydney Car Centre <Merrill8783@168city.com>,Gnitpick <gnitpick@flax9.uwaterloo.ca>,"Fri, 06 Jul 2007 06:59:51 -0400",the reply for your request for a job place [le...,\n\n\n\nWhile we may have high ...,1.0,1,TREC_07.csv
218083,Philippe Grosjean <phgrosjean@sciviews.org>,Duncan Murdoch <murdoch@stats.uwo.ca>,"Fri, 06 Jul 2007 12:57:17 +0200","Re: [R] Me again, about the horrible documenta...","For those who are interested, I just cook a li...",0.0,1,TREC_07.csv
218084,Bernhard Wellhöfer <Bernhard.Wellhoefer@gaia-g...,r-help@stat.math.ethz.ch,"Fri, 06 Jul 2007 12:43:12 +0200",Re: [R] RODBC problem,"Hello,\n\nas I wrote I call\n\n sqlFetch(chan...",0.0,1,TREC_07.csv


In [3]:
# Cell 4
print(data_df.columns.tolist())
print(data_df.info())

['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls', 'source_file']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 218086 entries, 0 to 218085
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   sender       184505 non-null  object 
 1   receiver     178821 non-null  object 
 2   date         182172 non-null  object 
 3   subject      214861 non-null  object 
 4   body         217466 non-null  object 
 5   label        217314 non-null  float64
 6   urls         184688 non-null  object 
 7   source_file  218086 non-null  object 
dtypes: float64(1), object(7)
memory usage: 13.3+ MB
None


In [4]:
# Cell 5
# stamdardize case and remove whitespace from column names
data_df.columns = (
    data_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [5]:
# Cell 6
#missing values
print(data_df[["subject", "body"]].isna().sum())

subject    3225
body        620
dtype: int64


In [6]:
# Cell 7
#replace missing with empty string
data_df["subject"] = data_df["subject"].fillna("")
data_df["body"] = data_df["body"].fillna("")

In [7]:
# Cell 8
missing_content = (
    (data_df["subject"].str.strip() == "") &
    (data_df["body"].str.strip() == "")
)

print("Emails with no subject or body:", missing_content.sum())

Emails with no subject or body: 617


<!-- Cell 9 -->
### Flagged Data to eliminate

In [8]:
# Cell 10
flagged_df = data_df[missing_content]
flagged_df.head() 

,sender,receiver,date,subject,body,label,urls,source_file
91972,有專人為你確認訂購單，所以請安心購買,1,1,,,NaN,NaN,TREC_05.csv
91986,����������.. http://www.freegostop.com,1,1,,,NaN,NaN,TREC_05.csv
92067,下完訂單將有專人再次為你確認，確認無誤後將在48小時內送到你府上 如要訂購 請填寫以...,None,None,,,NaN,NaN,TREC_05.csv
92068,\t\t 聯絡電話: 郵遞區號: 送達時間:,None,None,,,NaN,NaN,TREC_05.csv
92069,\t\t早上,None,None,,,NaN,NaN,TREC_05.csv


In [9]:
# Cell 11
# flagged rows are saved to data/flagged/flagged.csv in Cell 31,
# together with duplicates and rows with no label.
# (the file is rewritten on each run, so re-running doesn't add copies)

<!-- Cell 12 -->
#### Decode HTML

Email bodies may contain HTML marks and entities. 

In [10]:
# Cell 13
# before decode
def contains_html(text):
    if pd.isna(text):
        return False

    pattern = r"</?(html|body|head|div|p|br|a|span|table|tr|td|img|style|font|meta|title|h[1-6])\b[^>]*>"

    return bool(re.search(pattern, str(text), re.IGNORECASE))


html_emails = data_df[data_df["body"].apply(contains_html)]

print("Emails containing possible HTML:", len(html_emails))

display(html_emails[["subject", "body", "source_file"]].head(10))

Emails containing possible HTML: 36


,subject,body,source_file
23250,Replica Pens,Why would you want to purchase a replica watch...,CEAS_08.csv
72179,Your Apple ID was used to sign in to iMessage ...,Your Apple ID was used to sign in to \t\t\t\t...,Nazario.csv
73361,Fwd: Truly Irish........,\n\nReturn-path: <Comein63@aol.com>\nFrom: Com...,Nazario_5.csv
73374,PanNat Value - Confidential and Proprietary,"Jeff,\n\nPer your request, below is the value ...",Nazario_5.csv
73383,FW: Request for Confidential Information from ...,\n\n -----Original Message-----\nFrom: \tBenev...,Nazario_5.csv
73389,Cougars@Enron - New Interim Dean Named for CBA,"Rod,\nI wanted to forward this to you. Arthur...",Nazario_5.csv
73391,Cougars@Enron UH Alumni Group,Good afternoon.\n\nI wanted to forward you one...,Nazario_5.csv
73485,Your VerticalNet eBusiness Training Services L...,\n<P>\nYour login information for VerticalNet'...,Nazario_5.csv
73500,RE: Americas Gas and Power East doorstep reports,Christy/Shona\nPlease find attached the latest...,Nazario_5.csv
73501,RE: Americas Gas and Power East doorstep reports,"Christy,\nPse find attached the latest review ...",Nazario_5.csv


In [11]:
# Cell 14
def clean_email_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # decode HTML entities (including nested encoding)
    while True:
        decoded = html.unescape(text)

        if decoded == text:
            break

        text = decoded

    # keep links and email addresses written in angle brackets: <http://...>, <name@site.com>
    text = re.sub(r"<((?:https?://|mailto:)[^<>\s]+)>", r" \1 ", text, flags=re.IGNORECASE)
    text = re.sub(r"<([^<>\s]+@[^<>\s]+)>", r" \1 ", text)

    # keep the link inside <a href="..."> tags (useful for URL features)
    text = re.sub(r"<a\s[^>]*?href\s*=\s*[\"']?([^\"'\s>]+)[^>]*>", r" \1 ", text, flags=re.IGNORECASE)

    # Remove the remaining HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [12]:
# Cell 15
data_df["subject"] = data_df["subject"].apply(clean_email_text)
data_df["body"] = data_df["body"].apply(clean_email_text)

In [13]:
# Cell 16
data_df['body']

0         Buck up, your troubles caused by small dimensi...
1         Upgrade your sex and pleasures with these tech...
2         >+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...
3         Would anyone object to removing .so from this ...
4         WelcomeFastShippingCustomerSupport http://7iwf...
                                ...                        
218081    While we may have high expectations of our ass...
218082    While we may have high expectations of our ass...
218083    For those who are interested, I just cook a li...
218084    Hello, as I wrote I call sqlFetch(channel,"t_s...
218085    Well, are implicated. When does not the stage ...
Name: body, Length: 218086, dtype: object

In [14]:
# Cell 17
#check if html remains
#tags
html_remaining = data_df[
    data_df["body"].apply(contains_html)
]

print("Emails with remaining possible HTML tags:", len(html_remaining))


#html entities
entity_pattern = r"&(?:amp|lt|gt|quot|apos|nbsp);|&#\d+;|&#x[0-9a-fA-F]+;"

entity_remaining = data_df[
    data_df["body"].str.contains(
        entity_pattern,
        case=False,
        regex=True,
        na=False
    )
]

print("Emails with remaining HTML entities:", len(entity_remaining))

Emails with remaining possible HTML tags: 0
Emails with remaining HTML entities: 0


<!-- Cell 18 -->
## Text processing
lowercase / normalize text

In [15]:
# Cell 19
print(data_df.dtypes)

sender          object
receiver        object
date            object
subject         object
body            object
label          float64
urls            object
source_file     object
dtype: object


<!-- Cell 20 -->
.fillna('') – Replaces missing values with empty strings.



.astype(str) – Converts values into strings.



.str.lower() – Converts text to lowercase.



.str.strip() – Removes leading and trailing whitespace.

In [16]:
# Cell 21
#lowercase / strip whitespace / normalize
#will keep others preserved
columns = ['body', 'subject', 'sender', 'receiver']

for col in columns:
    data_df[col] = data_df[col].fillna("").astype(str).str.lower().str.strip()

#convert date
data_df['date'] = pd.to_datetime(
    data_df['date'],
    errors='coerce',
    utc=True
)

<!-- Cell 22 -->
#### URL feature extraction

In [17]:
# Cell 23
#make URL a string type
data_df['urls'] = (
    data_df['urls']
    .fillna('')
    .astype(str)
)

In [18]:
# Cell 24
data_df['urls'].unique()

array(['1', '0', '',
       "['http://web.bna.com', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5k3h4_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5e3e4_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5q2p6_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5j7f6_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j4v7g9_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5g8z1_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5h0x6_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5j8p4_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5h5h2_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5c8p3_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5m9f3_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5e3e4_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5k3h4_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5e3e4_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5q2p6_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5m7q5_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j4m8v2_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5m9

<!-- Cell 25 -->
The `urls` column mixes formats: `0`/`1`, `0.0`/`1.0`, and actual link lists (only Nazario_5). Enron and Ling have no URL info.

Steps:
1. **`url_flag`**: one consistent 0/1 column (NaN where the file has no URL info)
2. **`url_list`**: the actual links, found in each email's subject and body
3. **URL features**: numbers built from those links that a model can use

In [19]:
# Cell 26
# make the urls column one consistent format
# 1 = email has links, 0 = no links, NaN = file has no url info (Enron, Ling)
def normalize_url_flag(value):
    value = str(value).strip()
    if value in ("1", "1.0"):
        return 1
    if value in ("0", "0.0"):
        return 0
    if value.startswith("["):        # Nazario_5 stores the actual links as a list
        return 0 if value == "[]" else 1
    return np.nan                    # empty = no url info for this file


data_df["url_flag"] = data_df["urls"].apply(normalize_url_flag)
data_df["url_flag"].value_counts(dropna=False)

url_flag
1.0    100079
0.0     84609
NaN     33398
Name: count, dtype: int64

In [20]:
# Cell 27
# find the links written in each email (subject + body)
# only Nazario_5 lists its actual links, so we pull them out of the text for every file
import ast

URL_PATTERN = re.compile(r"(?:https?://|www\.)[^\s<>\"'()\[\]{}]+")


def find_urls(subject, body, urls):
    found = URL_PATTERN.findall(subject + " " + body)

    # add the links Nazario_5 already listed, e.g. "['http://a.com', 'http://b.com']"
    if urls.startswith("["):
        try:
            found += [str(u) for u in ast.literal_eval(urls)]
        except (ValueError, SyntaxError):
            pass

    # remove punctuation stuck to the end, lowercase, drop repeats (keeps order)
    cleaned = [u.rstrip(".,;:!?").lower() for u in found]
    return list(dict.fromkeys(cleaned))


data_df["url_list"] = [
    find_urls(s, b, u)
    for s, b, u in zip(data_df["subject"], data_df["body"], data_df["urls"])
]

data_df.loc[data_df["url_list"].str.len() > 0, ["url_list", "label"]].head()

,url_list,label
0,[http://whitedone.com/],1.0
1,[http://www.brightmade.com],1.0
2,[http://www.cnn.com/video/partners/email/index...,1.0
3,[http://en.wikipedia.org/wiki/.so_%28domain_na...,0.0
4,[http://7iwfna.blu.livefilestore.com/y1pxdx3kw...,1.0


In [21]:
# Cell 28
# build number features from each email's list of links
from urllib.parse import urlparse

SUSPICIOUS_WORDS = ["login", "signin", "verify", "account", "update",
                    "secure", "bank", "confirm", "password", "paypal"]
SHORTENERS = {"bit.ly", "tinyurl.com", "goo.gl", "t.co", "ow.ly", "is.gd", "buff.ly"}
IP_PATTERN = re.compile(r"^\d{1,3}(?:\.\d{1,3}){3}$")   # e.g. 192.168.0.1


def get_domain(url):
    # "http://www.paypal.com/login" -> "www.paypal.com"
    if url.startswith("www."):
        url = "http://" + url
    try:
        return urlparse(url).hostname or ""
    except ValueError:
        return ""


def url_features(urls):
    domains = [get_domain(u) for u in urls]
    return {
        "url_count": len(urls),                                                      # number of links
        "url_unique_domains": len({d for d in domains if d}),                        # different websites linked
        "url_max_length": max((len(u) for u in urls), default=0),                    # longest link
        "url_has_ip": int(any(IP_PATTERN.match(d) for d in domains)),                # link uses an IP instead of a name
        "url_has_http_only": int(any(u.startswith("http://") for u in urls)),        # link without https
        "url_has_at_symbol": int(any("@" in u for u in urls)),                       # "@" can hide the real site
        "url_has_shortener": int(any(d.removeprefix("www.") in SHORTENERS for d in domains)),  # bit.ly etc.
        "url_has_suspicious_word": int(any(w in u for u in urls for w in SUSPICIOUS_WORDS)),   # login, verify...
        "url_max_subdomains": max((d.count(".") for d in domains), default=0),       # dots in the website name
    }


url_feats = pd.DataFrame([url_features(u) for u in data_df["url_list"]], index=data_df.index)
url_feature_cols = list(url_feats.columns)

data_df = pd.concat([data_df, url_feats], axis=1)

# turn the lists back into text (links separated by spaces)
# so drop_duplicates() and saving to CSV work later
data_df["url_list"] = data_df["url_list"].str.join(" ")

data_df[url_feature_cols].describe().round(2)

,url_count,url_unique_domains,url_max_length,url_has_ip,url_has_http_only,url_has_at_symbol,url_has_shortener,url_has_suspicious_word,url_max_subdomains
count,218086.0,218086.00,218086.00,218086.00,218086.00,218086.00,218086.00,218086.00,218086.00
mean,1.3,0.70,20.28,0.00,0.40,0.02,0.00,0.01,0.83
std,8.5,5.89,33.38,0.04,0.49,0.15,0.02,0.11,1.06
min,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
50%,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
75%,1.0,1.00,35.00,0.00,1.00,0.00,0.00,0.00,2.00
max,3134.0,2398.00,3632.00,1.00,1.00,1.00,1.00,1.00,26.00


In [22]:
# Cell 29
# check: does the original url flag agree with the links we found?
print(pd.crosstab(data_df["url_flag"], data_df["url_count"] > 0, dropna=False))

# average of each URL feature for legit (0) vs phishing/spam (1)
data_df.groupby("label")[url_feature_cols].mean().round(3).T

url_count  False  True 
url_flag               
0.0        81197   3412
1.0        12254  87825
NaN        33388     10


label,0.0,1.0
url_count,1.308,1.311
url_unique_domains,0.859,0.538
url_max_length,21.079,19.560
url_has_ip,0.001,0.002
url_has_http_only,0.377,0.423
url_has_at_symbol,0.009,0.040
url_has_shortener,0.001,0.000
url_has_suspicious_word,0.008,0.016
url_max_subdomains,0.885,0.766


<!-- Cell 30 -->
### Eliminate flagged rows

- emails with no subject and no body
- emails with no label (broken rows from reading the CSV)
- exact duplicates (keeps the first copy)

In [23]:
# Cell 31
#remove flagged from main df
# (checks run on the current data, after cleaning, so nothing gets out of sync)
no_content = (data_df["subject"].str.strip() == "") & (data_df["body"].str.strip() == "")
no_label = data_df["label"].isna()
duplicate = data_df.duplicated(keep="first")

print("no subject or body:", no_content.sum())
print("no label:", no_label.sum())
print("duplicates:", duplicate.sum())

# one reason per row (a row can match more than one, the first match is used)
flag_reason = np.select(
    [no_content, no_label, duplicate],
    ["no_content", "no_label", "duplicate"],
    default=""
)
is_flagged = flag_reason != ""

#save every removed row, with its reason, to flagged.csv (rewritten each run)
flagged_dir = PROJECT_ROOT / "data" / "flagged"
flagged_dir.mkdir(parents=True, exist_ok=True)
removed = data_df[is_flagged].assign(flag_reason=flag_reason[is_flagged])
removed.to_csv(flagged_dir / "flagged.csv", index=False)

data_df = data_df[~is_flagged]

print("\nrows removed:", len(removed))
print(removed["flag_reason"].value_counts())

no subject or body: 617
no label: 772
duplicates: 525

rows removed: 1181
flag_reason
no_content    617
duplicate     409
no_label      155
Name: count, dtype: int64


<!-- Cell 32 -->
#### New cleaned dataset

In [24]:
# Cell 33
#preserve original
print(data_df["source_file"].value_counts())
print("\n\nFinal shape:", data_df.shape)

source_file
TREC_05.csv           55234
TREC_07.csv           53755
CEAS_08.csv           39154
Enron.csv             29424
TREC_06.csv           16395
Nigerian_5.csv         6325
SpamAssasin.csv        5808
Nigerian_Fraud.csv     3328
Nazario_5.csv          3061
Ling.csv               2859
Nazario.csv            1562
Name: count, dtype: int64


Final shape: (216905, 19)


In [25]:
# Cell 34
#exporting cleaned dataset
import os

os.makedirs("../data/processed", exist_ok=True)
data_df.to_csv(
    "../data/processed/cleaned_emails.csv",
    index=False
)